# MATLAB-Literal GUI Parity Against the 180 s Export

This notebook compares the MATLAB GUI export in
`results/source_exports/gui_key_signals_export.csv` against a fresh
standalone `simuoriginal_replica` run using:

- duration = `180 s`
- environment switch = `30 s`
- `Fe` mode = `gui_skin_locked`
- initial state = zero position
- force input = sine, amplitude `5 N`, bias `5 N`, frequency `0.5 rad/s`

The legacy `gui_key_signals.csv` remains a `30 s` reference only.

The MATLAB GUI export starts from zero position and zero force, so the Python
parity run below uses the same zero-position initial state explicitly.


In [ ]:
import csv
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    for root in (candidate, candidate / 'TeleopWithRL'):
        if (root / 'matlab_env_python_replica').exists() and (root / 'notebooks' / '_teleop_nb.py').exists():
            if str(root) not in sys.path:
                sys.path.insert(0, str(root))
            break
    else:
        continue
    break
else:
    raise RuntimeError('Could not find TeleopWithRL notebook root.')

from matlab_env_python_replica.simuoriginal_replica import (
    FE_MODE_GUI,
    ParmsOriginal,
    build_saved_simuoriginal_state,
    SimuOriginalProfile,
    saved_force_input,
    simulate_simuoriginal_replica,
    write_simuoriginal_result,
)
from notebooks._teleop_nb import repo_paths, show_rows

plt.rcParams['figure.dpi'] = 120

P = repo_paths()
RESULTS = P['matlab_results']
SOURCE_EXPORTS = RESULTS / 'source_exports'
GUI_SOURCE = SOURCE_EXPORTS / 'gui_key_signals_export.csv'
LEGACY_GUI_SOURCE = SOURCE_EXPORTS / 'gui_key_signals.csv'
RUN_ROOT = RESULTS / 'io_parity_run_gui_export_180s_switch30s'
SIM_OUT = RUN_ROOT / 'simuoriginal'
COMPARE_ROOT = RESULTS / 'matlab_vs_simuoriginal_gui_export_180s_switch30s'
PLOTS = COMPARE_ROOT / 'plots'
ALIGNED_PATH = COMPARE_ROOT / 'matlab_vs_simuoriginal_aligned.csv'
METRICS_PATH = COMPARE_ROOT / 'comparison_metrics.txt'
CFG = {
    'duration_s': 180.0,
    'env_switch_time_s': 30.0,
    'force_amp_N': 5.0,
    'force_bias_N': 5.0,
    'force_freq_rad_s': 0.5,
    'force_phase_rad': 0.0,
    'fe_mode': FE_MODE_GUI,
    'init_position_mode': 'zero',
}

def load_numeric_csv(path: Path) -> dict[str, np.ndarray]:
    with path.open('r', encoding='utf-8', newline='') as fh:
        rows = list(csv.DictReader(fh))
    cols = {key: np.array([float(row[key]) for row in rows], dtype=float) for key in rows[0].keys()}
    return cols

def rmse(values: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(values)))) if values.size else float('nan')

def max_abs(values: np.ndarray) -> float:
    return float(np.max(np.abs(values))) if values.size else float('nan')

def corr(a: np.ndarray, b: np.ndarray) -> float:
    if a.size < 2 or b.size < 2:
        return float('nan')
    return float(np.corrcoef(a, b)[0, 1])

show_rows(
    [
        {'artifact': 'gui_source_180s', 'path': str(GUI_SOURCE)},
        {'artifact': 'legacy_gui_source_30s', 'path': str(LEGACY_GUI_SOURCE)},
        {'artifact': 'standalone_export_dir', 'path': str(SIM_OUT)},
        {'artifact': 'compare_root', 'path': str(COMPARE_ROOT)},
        *({'artifact': key, 'path': value} for key, value in {
            'duration_s': CFG['duration_s'],
            'env_switch_time_s': CFG['env_switch_time_s'],
            'force_amp_N': CFG['force_amp_N'],
            'force_bias_N': CFG['force_bias_N'],
            'force_freq_rad_s': CFG['force_freq_rad_s'],
            'fe_mode': CFG['fe_mode'],
        }.items()),
    ],
    title='Parity inputs and outputs',
    max_rows=20,
)


In [ ]:
gui = load_numeric_csv(GUI_SOURCE)
gui_rows = int(gui['t'].size)
gui_dt = float(gui['t'][1] - gui['t'][0]) if gui_rows > 1 else float('nan')
gui_resets = int(np.sum(np.diff(gui['t']) < 0.0))

parms = ParmsOriginal()
profile = SimuOriginalProfile(
    fixed_step=parms.Ts,
    force_amplitude=CFG['force_amp_N'],
    force_bias=CFG['force_bias_N'],
    force_frequency_rad=CFG['force_freq_rad_s'],
    force_phase_rad=CFG['force_phase_rad'],
    env_switch_time=CFG['env_switch_time_s'],
)
force_fn = lambda t: saved_force_input(t, profile)
control_fn = lambda _t: 0.0
initial_state = build_saved_simuoriginal_state(parms, init_position_mode=CFG['init_position_mode'])

result = simulate_simuoriginal_replica(
    duration=CFG['duration_s'],
    parms=parms,
    profile=profile,
    initial_state=initial_state,
    F_h_fn=force_fn,
    u_fn=control_fn,
    fe_mode=CFG['fe_mode'],
)
SIM_OUT.mkdir(parents=True, exist_ok=True)
write_simuoriginal_result(result, SIM_OUT)

gui_t = gui['t']
sim_t = np.asarray(result.time, dtype=float)
sim_end = float(sim_t[-1])
gui_end = float(gui_t[-1])
overlap_mask = gui_t <= (sim_end + 1e-12)
aligned_t = gui_t[overlap_mask]

sim_interp = {
    'x_m': np.interp(aligned_t, sim_t, np.asarray(result.x_m, dtype=float)),
    'x_s': np.interp(aligned_t, sim_t, np.asarray(result.x_s, dtype=float)),
    'Fe': np.interp(aligned_t, sim_t, np.asarray(result.Fe, dtype=float)),
    'x_mdot': np.interp(aligned_t, sim_t, np.asarray(result.xm_dot, dtype=float)),
    'x_sdot': np.interp(aligned_t, sim_t, np.asarray(result.xs_dot, dtype=float)),
}
gui_overlap = {
    'x_m': gui['x_m'][overlap_mask],
    'x_s': gui['x_s'][overlap_mask],
    'Fe': gui['Fe'][overlap_mask],
    'x_mdot': gui['x_mdot'][overlap_mask],
    'x_sdot': gui['x_sdot'][overlap_mask],
}
err = {key: sim_interp[key] - gui_overlap[key] for key in sim_interp}

PLOTS.mkdir(parents=True, exist_ok=True)
COMPARE_ROOT.mkdir(parents=True, exist_ok=True)

with ALIGNED_PATH.open('w', encoding='utf-8', newline='') as fh:
    fieldnames = [
        't',
        'matlab_x_s', 'simuoriginal_x_s', 'err_x_s',
        'matlab_x_m', 'simuoriginal_x_m', 'err_x_m',
        'matlab_Fe', 'simuoriginal_Fe', 'err_Fe',
        'matlab_x_mdot', 'simuoriginal_x_mdot', 'err_x_mdot',
        'matlab_x_sdot', 'simuoriginal_x_sdot', 'err_x_sdot',
    ]
    writer = csv.DictWriter(fh, fieldnames=fieldnames)
    writer.writeheader()
    for idx, t_value in enumerate(aligned_t):
        writer.writerow({
            't': float(t_value),
            'matlab_x_s': float(gui_overlap['x_s'][idx]),
            'simuoriginal_x_s': float(sim_interp['x_s'][idx]),
            'err_x_s': float(err['x_s'][idx]),
            'matlab_x_m': float(gui_overlap['x_m'][idx]),
            'simuoriginal_x_m': float(sim_interp['x_m'][idx]),
            'err_x_m': float(err['x_m'][idx]),
            'matlab_Fe': float(gui_overlap['Fe'][idx]),
            'simuoriginal_Fe': float(sim_interp['Fe'][idx]),
            'err_Fe': float(err['Fe'][idx]),
            'matlab_x_mdot': float(gui_overlap['x_mdot'][idx]),
            'simuoriginal_x_mdot': float(sim_interp['x_mdot'][idx]),
            'err_x_mdot': float(err['x_mdot'][idx]),
            'matlab_x_sdot': float(gui_overlap['x_sdot'][idx]),
            'simuoriginal_x_sdot': float(sim_interp['x_sdot'][idx]),
            'err_x_sdot': float(err['x_sdot'][idx]),
        })


def metric_row(segment: str, mask: np.ndarray) -> dict:
    return {
        'segment': segment,
        'rows': int(np.count_nonzero(mask)),
        't_start_s': float(aligned_t[mask][0]) if np.any(mask) else float('nan'),
        't_end_s': float(aligned_t[mask][-1]) if np.any(mask) else float('nan'),
        'duration_s': float(aligned_t[mask][-1] - aligned_t[mask][0]) if np.count_nonzero(mask) > 1 else 0.0,
        'x_m_rmse': rmse(err['x_m'][mask]),
        'x_s_rmse': rmse(err['x_s'][mask]),
        'Fe_rmse': rmse(err['Fe'][mask]),
        'x_mdot_rmse': rmse(err['x_mdot'][mask]),
        'x_sdot_rmse': rmse(err['x_sdot'][mask]),
    }


full_mask = np.ones_like(aligned_t, dtype=bool)
pre_mask = aligned_t < CFG['env_switch_time_s']
post_mask = aligned_t >= CFG['env_switch_time_s']
metric_rows = [
    metric_row('full_overlap', full_mask),
    metric_row('pre_switch_overlap', pre_mask),
    metric_row('post_switch_overlap', post_mask),
]

source_rows = [
    {
        'gui_rows': gui_rows,
        'gui_t_last_s': gui_end,
        'gui_dt_s': gui_dt,
        'gui_num_resets': gui_resets,
        'sim_rows': int(sim_t.size),
        'sim_t_last_s': sim_end,
        'sim_singularity_time_s': result.singularity_time,
        'overlap_rows': int(aligned_t.size),
        'overlap_t_last_s': float(aligned_t[-1]),
        'gui_tail_unmatched_s': float(gui_end - sim_end),
        'gui_coverage_pct': float(100.0 * sim_end / gui_end),
    }
]

with METRICS_PATH.open('w', encoding='utf-8') as fh:
    fh.write('MATLAB GUI export vs standalone simuoriginal_replica\n')
    fh.write(f'gui_source: {GUI_SOURCE}\n')
    fh.write(f'legacy_gui_source: {LEGACY_GUI_SOURCE}\n')
    fh.write(f'standalone_out_dir: {SIM_OUT}\n')
    fh.write(f'duration_s_requested: {CFG["duration_s"]}\n')
    fh.write(f'env_switch_time_s: {CFG["env_switch_time_s"]}\n')
    fh.write(f'force_amp_N: {CFG["force_amp_N"]}\n')
    fh.write(f'force_bias_N: {CFG["force_bias_N"]}\n')
    fh.write(f'force_freq_rad_s: {CFG["force_freq_rad_s"]}\n')
    fh.write(f'fe_mode: {CFG["fe_mode"]}\n')
    fh.write(f'init_position_mode: {CFG["init_position_mode"]}\n')
    fh.write(f'gui_rows: {gui_rows}\n')
    fh.write(f'gui_t_last_s: {gui_end}\n')
    fh.write(f'gui_dt_s: {gui_dt}\n')
    fh.write(f'gui_num_resets: {gui_resets}\n')
    fh.write(f'sim_rows: {int(sim_t.size)}\n')
    fh.write(f'sim_t_last_s: {sim_end}\n')
    fh.write(f'sim_singularity_time_s: {result.singularity_time}\n')
    fh.write(f'overlap_rows: {int(aligned_t.size)}\n')
    fh.write(f'overlap_t_last_s: {float(aligned_t[-1])}\n')
    fh.write(f'gui_tail_unmatched_s: {float(gui_end - sim_end)}\n')
    fh.write('\n')
    for row in metric_rows:
        fh.write(f'[{row["segment"]}]\n')
        for key, value in row.items():
            if key == 'segment':
                continue
            fh.write(f'{key}: {value}\n')
        fh.write('\n')
    fh.write('[correlation_full_overlap]\n')
    for key in ('x_m', 'x_s', 'Fe', 'x_mdot', 'x_sdot'):
        fh.write(f'{key}_corr: {corr(gui_overlap[key], sim_interp[key])}\n')
        fh.write(f'{key}_max_abs_err: {max_abs(err[key])}\n')

show_rows(source_rows, title='Source and overlap summary', max_rows=10)
show_rows(metric_rows, title='Parity metrics by segment', max_rows=10)

full_overlay_path = PLOTS / 'overlay_full_run.png'
fig, axes = plt.subplots(5, 1, figsize=(14, 15), sharex=True)
plot_specs = [
    ('x_m', 'Master position x_m', 'm'),
    ('x_s', 'Slave position x_s', 'm'),
    ('Fe', 'Environment force F_e', 'N'),
    ('x_mdot', 'Master velocity x_mdot', 'm/s'),
    ('x_sdot', 'Slave velocity x_sdot', 'm/s'),
]
for idx, (key, title, ylabel) in enumerate(plot_specs):
    ax = axes[idx]
    ax.plot(gui_t, gui[key], lw=2.0, label='MATLAB GUI export', color='tab:blue')
    ax.plot(aligned_t, sim_interp[key], lw=1.8, label='Python replica', color='tab:orange')
    ax.axvline(CFG['env_switch_time_s'], color='0.35', lw=1.0, ls='--', alpha=0.8)
    if sim_end < gui_end:
        ax.axvline(sim_end, color='tab:red', lw=1.0, ls=':', alpha=0.85)
        ax.axvspan(sim_end, gui_end, color='0.92', alpha=0.8)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)
axes[0].legend(loc='upper right', ncol=2)
axes[-1].set_xlabel('time [s]')
axes[-1].set_xlim(0.0, gui_end)
fig.suptitle('Full-run GUI vs Python parity (tail shading appears only if Python stops early)')
fig.tight_layout()
fig.savefig(full_overlay_path, dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)

error_path = PLOTS / 'error_overlap.png'
fig, axes = plt.subplots(5, 1, figsize=(14, 15), sharex=True)
err_specs = [
    ('x_m', 'Master position error', 'm'),
    ('x_s', 'Slave position error', 'm'),
    ('Fe', 'Environment force error', 'N'),
    ('x_mdot', 'Master velocity error', 'm/s'),
    ('x_sdot', 'Slave velocity error', 'm/s'),
]
for idx, (key, title, ylabel) in enumerate(err_specs):
    ax = axes[idx]
    ax.plot(aligned_t, err[key], lw=1.5, color='tab:red')
    ax.axvline(CFG['env_switch_time_s'], color='0.35', lw=1.0, ls='--', alpha=0.8)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)
axes[-1].set_xlabel('time [s]')
axes[-1].set_xlim(0.0, sim_end)
fig.suptitle('Overlap-only error traces on the GUI time grid')
fig.tight_layout()
fig.savefig(error_path, dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)

zoom_path = PLOTS / 'overlay_switch_window.png'
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
zoom_specs = [
    ('x_m', 'Master position x_m', 'm'),
    ('x_s', 'Slave position x_s', 'm'),
    ('Fe', 'Environment force F_e', 'N'),
]
for idx, (key, title, ylabel) in enumerate(zoom_specs):
    ax = axes[idx]
    ax.plot(gui_t, gui[key], lw=2.0, label='MATLAB GUI export', color='tab:blue')
    ax.plot(aligned_t, sim_interp[key], lw=1.8, label='Python replica', color='tab:orange')
    ax.axvline(CFG['env_switch_time_s'], color='0.35', lw=1.0, ls='--', alpha=0.8)
    ax.axvline(sim_end, color='tab:red', lw=1.0, ls=':')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)
axes[0].legend(loc='upper right', ncol=2)
axes[-1].set_xlabel('time [s]')
axes[-1].set_xlim(20.0, min(gui_end, max(40.0, sim_end + 0.5)))
fig.suptitle('Switch-window view around the 30 s environment change')
fig.tight_layout()
fig.savefig(zoom_path, dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)

show_rows(
    [
        {'artifact': 'aligned_csv', 'path': str(ALIGNED_PATH)},
        {'artifact': 'metrics_txt', 'path': str(METRICS_PATH)},
        {'artifact': 'full_overlay_plot', 'path': str(full_overlay_path)},
        {'artifact': 'error_plot', 'path': str(error_path)},
        {'artifact': 'switch_window_plot', 'path': str(zoom_path)},
    ],
    title='Generated parity artifacts',
    max_rows=10,
)
